In [1]:
# Author: Om Mohite

import os
from pathlib import Path
from getpass import getpass

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

# Find and load the existing .env file
env_file = None

for folder in [Path.cwd(), *Path.cwd().parents]:
    candidates = [
        folder / ".env",
        folder / "Data-Engineering-Real-World-Projects" / ".env",
    ]

    for candidate in candidates:
        if candidate.exists():
            env_file = candidate
            break

    if env_file:
        load_dotenv(env_file)
        break

db_password = os.getenv("DB_PASSWORD") or getpass("MySQL password: ")

# Connect to MySQL server
server_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=db_password,
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "3306")),
)

server_engine = create_engine(server_url, pool_pre_ping=True)

with server_engine.begin() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS test_case_7"))

engine = create_engine(
    server_url.set(database="test_case_7"),
    pool_pre_ping=True,
)

create_table_statements = [
    """
    CREATE TABLE IF NOT EXISTS customers (
        customer_id INT PRIMARY KEY,
        customer_name VARCHAR(100),
        email VARCHAR(100),
        city VARCHAR(50),
        region VARCHAR(50),
        registration_date DATE
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS products (
        product_id INT PRIMARY KEY,
        product_name VARCHAR(100),
        category VARCHAR(50),
        unit_price DECIMAL(10, 2),
        stock_quantity INT
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS sales (
        sale_id INT PRIMARY KEY,
        customer_id INT,
        product_id INT,
        sale_date DATE,
        quantity INT,
        total_amount DECIMAL(12, 2),
        payment_status VARCHAR(30),
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    )
    """,
]

with engine.begin() as conn:
    for statement in create_table_statements:
        conn.execute(text(statement))

    tables = conn.execute(text("SHOW TABLES")).all()

print("Connected database: test_case_7")
print("Tables:", tables)

server_engine.dispose()

Connected database: test_case_7
Tables: [('customers',), ('products',), ('sales',)]


In [3]:
# Author: Om Mohite

import re
from pathlib import Path
from sqlalchemy import text

sql_candidates = [
    Path.cwd() / "test_case_7.sql",
    Path.cwd() / "Data-Engineering-Real-World-Projects" / "Test_Case_7" / "test_case_7.sql",
]

sql_file = next((path for path in sql_candidates if path.exists()), None)

if sql_file is None:
    raise FileNotFoundError("Could not find test_case_7.sql.")

sql_script = sql_file.read_text(encoding="utf-8")

table_names = ["customers", "products", "sales"]
insert_statements = {}

for table_name in table_names:
    statements = [
        statement.strip()
        for statement in sql_script.split(";")
        if re.match(
            rf"\s*INSERT\s+INTO\s+{table_name}\b",
            statement,
            re.IGNORECASE,
        )
    ]

    if not statements:
        raise ValueError(f"No INSERT statement found for {table_name}.")

    insert_statements[table_name] = statements

with engine.begin() as conn:
    # Clear test records in foreign-key-safe order
    conn.execute(text("DELETE FROM sales"))
    conn.execute(text("DELETE FROM customers"))
    conn.execute(text("DELETE FROM products"))

    for table_name in table_names:
        for statement in insert_statements[table_name]:
            conn.exec_driver_sql(statement)

    row_counts = {
        table_name: conn.execute(
            text(f"SELECT COUNT(*) FROM {table_name}")
        ).scalar_one()
        for table_name in table_names
    }

print("Rows loaded:", row_counts)

if row_counts != {"customers": 10, "products": 12, "sales": 30}:
    raise ValueError(f"Unexpected source row counts: {row_counts}")

Rows loaded: {'customers': 10, 'products': 12, 'sales': 30}


In [4]:
# Author: Om Mohite

def data_extraction(engine):
    customers_df = pd.read_sql(
        "SELECT * FROM customers",
        con=engine,
    )

    products_df = pd.read_sql(
        "SELECT * FROM products",
        con=engine,
    )

    sales_df = pd.read_sql(
        "SELECT * FROM sales",
        con=engine,
    )

    return customers_df, products_df, sales_df


customers, products, sales = data_extraction(engine)

print("Customers:", len(customers))
print("Products:", len(products))
print("Sales:", len(sales))

display(customers.head(3))
display(products.head(3))
display(sales.head(3))

Customers: 10
Products: 12
Sales: 30


,customer_id,customer_name,email,city,region,registration_date
0,1,Rahul Sharma,rahul@gmail.com,Pune,West,2023-01-15
1,2,Amit Verma,amit@gmail.com,Delhi,North,2023-02-20
2,3,Priya Patel,priya@gmail.com,Mumbai,West,2023-03-10


,product_id,product_name,category,unit_price,stock_quantity
0,101,Laptop,Electronics,65000.0,20
1,102,Smartphone,Electronics,25000.0,50
2,103,Headphones,Accessories,2500.0,100


,sale_id,customer_id,product_id,sale_date,quantity,total_amount,payment_status
0,1001,1,101,2024-01-05,1,65000.0,Completed
1,1002,2,102,2024-01-10,2,50000.0,Completed
2,1003,3,103,2024-01-15,3,7500.0,Completed


In [5]:
# Author: Om Mohite

join_query = """
    SELECT
        c.customer_id,
        p.product_id,
        s.sale_id,
        c.customer_name,
        c.email,
        c.city,
        c.region,
        c.registration_date,
        p.product_name,
        p.category,
        p.unit_price,
        p.stock_quantity,
        s.sale_date,
        s.quantity,
        s.total_amount,
        s.payment_status
    FROM customers AS c
    RIGHT JOIN sales AS s
        ON c.customer_id = s.customer_id
    RIGHT JOIN products AS p
        ON p.product_id = s.product_id
"""

complete_table = pd.read_sql(join_query, con=engine)

print("Joined rows:", len(complete_table))
display(complete_table.head())

Joined rows: 30


,customer_id,product_id,sale_id,customer_name,email,city,region,registration_date,product_name,category,unit_price,stock_quantity,sale_date,quantity,total_amount,payment_status
0,1,101,1001,Rahul Sharma,rahul@gmail.com,Pune,West,2023-01-15,Laptop,Electronics,65000.0,20,2024-01-05,1,65000.0,Completed
1,4,101,1014,Sneha Desai,sneha@gmail.com,Bangalore,South,2023-04-05,Laptop,Electronics,65000.0,20,2024-05-01,1,65000.0,Pending
2,3,101,1023,Priya Patel,priya@gmail.com,Mumbai,West,2023-03-10,Laptop,Electronics,65000.0,20,2024-07-12,1,65000.0,Completed
3,10,101,1030,Pooja Shah,pooja@gmail.com,Ahmedabad,West,2023-10-11,Laptop,Electronics,65000.0,20,2024-09-15,1,65000.0,Completed
4,2,102,1002,Amit Verma,amit@gmail.com,Delhi,North,2023-02-20,Smartphone,Electronics,25000.0,50,2024-01-10,2,50000.0,Completed


In [6]:
# Author: Om Mohite

region_sales = (
    complete_table
    .groupby("region")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

print("Sales by region")
display(region_sales.to_frame("total_sales"))

Sales by region


,total_sales
region,
West,359000.0
North,171000.0
South,140700.0
Central,44000.0


In [7]:
# Author: Om Mohite

source_sales_count = len(sales)
joined_row_count = len(complete_table)
unique_joined_sales = complete_table["sale_id"].nunique()

source_total = float(sales["total_amount"].sum())
joined_total = float(complete_table["total_amount"].sum())
regional_total = float(region_sales.sum())

validation_report = {
    "source_sales_rows": source_sales_count,
    "joined_rows": joined_row_count,
    "unique_joined_sales": unique_joined_sales,
    "source_total_amount": source_total,
    "joined_total_amount": joined_total,
    "region_report_total": regional_total,
}

display(pd.Series(validation_report, name="value").to_frame())

assert joined_row_count == source_sales_count
assert unique_joined_sales == source_sales_count
assert abs(source_total - joined_total) < 0.01
assert abs(source_total - regional_total) < 0.01

print("TEST CASE 7 VALIDATION PASSED")

,value
source_sales_rows,30.0
joined_rows,30.0
unique_joined_sales,30.0
source_total_amount,714700.0
joined_total_amount,714700.0
region_report_total,714700.0


TEST CASE 7 VALIDATION PASSED
